# ZivaWhite on a GPU: tokenizer -> pretrain -> chat -> tools -> preferences -> export

Run top to bottom on any CUDA GPU (Colab, a workstation, a rented box). **Nothing here is capped by your hardware**:
model size comes from the config you pick. The first cell estimates what your GPU can afford, so you choose with numbers.

**Read this first.** A model trained from scratch in one session is a *learning* model: it will be fluent in places and
wrong often. For Ollama-class behaviour, start from open weights instead (see "Option B" below). Details and honest
status of every step: `docs/CAPABILITIES.md`. The code in this notebook was written without being able to execute the
torch parts, so **run `pytest -q` (cell 2) first** and read any failure before spending GPU hours.

## 0. Setup

In [ ]:
# Upload ziva-white.zip to the session (or clone your repo), then:
!unzip -q -o ziva-white.zip
%cd ziva-white
!pip install -q -r requirements.txt -r requirements-publish.txt pytest httpx fastapi uvicorn
!nvidia-smi

In [ ]:
# Run the test suite on THIS machine (torch, fastapi, transformers are installed now). All of it should pass.
!python -m pytest -q -x 2>&1 | tail -15

In [ ]:
from zivawhite.device import resolve_runtime
print(resolve_runtime().summary())          # device + precision only; it never decides your model size

## 1. What can this GPU afford?  (advice only; change the numbers freely)

In [ ]:
GPU = "a100"        # t4 l4 a10 rtx3090 rtx4090 a6000 l40s a100-40 a100 h100-pcie h100   (or pass --tflops in the command)
N_GPUS = 1
HOURS = 12
!python -m zivawhite.app plan --gpu {GPU} --n_gpus {N_GPUS} --hours {HOURS}

## 2. A general tokenizer

The bundled 12k tokenizer is English books. A general model wants a larger vocabulary trained on broad text.
This streams a sample of FineWeb (needs `datasets`; ~a few minutes).
**Check the license notes it prints** (also written to `*.provenance.json`).

In [ ]:
!python -m zivawhite.app data tokenizer --recipe fineweb --max_docs 200k --vocab_size 32000 --out tokenizers/gen32k

## 3. Pretraining data (streamed straight into shards; memory stays flat)

In [ ]:
TOKENS = "300m"      # raise it as far as your time budget allows; the plan above says how long it takes
!python -m zivawhite.app data text --recipe fineweb --tokenizer_dir tokenizers/gen32k --out_dir data/processed --max_tokens {TOKENS}

## 4. Pretrain

`configs/general_125m.yaml` is a ~125M model (16 layers, 768 wide, grouped-query attention). Change any number on the
command line, or use `general_300m.yaml` / `general_1b.yaml`. Use `--max_steps` to fit your budget.
`--grad_ckpt` trades ~30% speed for much more memory headroom. A warning about memory is advice, never a limit.

In [ ]:
!python -m zivawhite.app train --config configs/general_125m.yaml --data_dir data/processed \
        --tokenizer_dir tokenizers/gen32k --out_dir runs/base --max_steps 2000

### Option B: start from open weights instead (the route to Ollama-class quality)

Skip steps 2-4 and import a plain Llama-architecture model (e.g. SmolLM2, Apache-2.0). Then pass
`--tokenizer_dir runs/base/tokenizer` in the later steps. The weights keep their original license.

In [ ]:
# !huggingface-cli download HuggingFaceTB/SmolLM2-360M --local-dir models/smollm2-360m
# !python -m zivawhite.app import-hf --src models/smollm2-360m --out runs/base

## 5. Chat data: smol-smoltalk (built for sub-1B models) + verifiable tool-use conversations

In [ ]:
!python -m zivawhite.app data chat --recipe smol-smoltalk --out data/chat/smol.jsonl --max_rows 50k
!python -m zivawhite.app tool-data --out data/chat/tool_use.jsonl --n 3000

## 6. Supervised fine-tuning

Learns only the assistant's replies, with system/tool roles, epochs, a validation split and early stopping.
Replace `--tokenizer_dir` with `runs/base/tokenizer` if you imported a model.

In [ ]:
TOK = "tokenizers/gen32k"
!python -m zivawhite.app sft --data data/chat/smol.jsonl data/chat/tool_use.jsonl --init_from runs/base/checkpoint.pt \
        --tokenizer_dir {TOK} --out_dir runs/chat --epochs 2 --batch_size 8

## 7. Does it work? Measure before and after, not by feel

In [ ]:
# held-out text the model never trained on: put some in data/heldout.txt
# !python -m zivawhite.app eval --checkpoint runs/chat/checkpoint.pt --tokenizer_dir {TOK} --text data/heldout.txt
open("prompts.txt", "w").write("Hello! Who are you?\nExplain what a neural network is in two sentences.\nWhat is 17% of 2340?\n")
!python -m zivawhite.app eval --checkpoint runs/chat/checkpoint.pt --tokenizer_dir {TOK} --prompts prompts.txt --repetition_penalty 1.1

In [ ]:
!python -m zivawhite.app ask "Hello! Who are you?" --checkpoint runs/chat/checkpoint.pt --route local
!python -m zivawhite.app agent "What is 17% of 2340?" --checkpoint runs/chat/checkpoint.pt

## 8. Preference tuning (DPO): a second stage

Only worthwhile on a model that already chats. `--sft_alpha 0.1` stops DPO from lowering the likelihood of good answers.
UltraFeedback's completions come from a mix of open and proprietary models: read its provenance note before publishing.

In [ ]:
!python -m zivawhite.app data prefs --recipe ultrafeedback --out data/prefs/uf.jsonl --max_rows 10k
!python -m zivawhite.app dpo --pairs data/prefs/uf.jsonl --init_from runs/chat/checkpoint.pt --tokenizer_dir {TOK} \
        --sft_alpha 0.1 --epochs 1 --out_dir runs/dpo
# compare against runs/chat with the eval cell above; keep whichever scores better

## 9. Export, verify, publish

In [ ]:
!python -m zivawhite.app export --checkpoint runs/dpo/checkpoint_best.pt --tokenizer_dir {TOK} \
        --out exports/my-model --name My-Model --verify
# Review exports/my-model/README.md (training data + license!). Then, with a write token in HF_TOKEN:
# import os; os.environ["HF_TOKEN"] = "hf_..."   # or use Colab secrets
# !python -m zivawhite.app export --checkpoint runs/dpo/checkpoint_best.pt --tokenizer_dir {TOK} --out exports/my-model --push YOUR_NAME/My-Model

## 10. Use it elsewhere

* **Ollama:** `cd exports/my-model && ollama create my-model -f Modelfile` (see `docs/PUBLISHING.md`, including the
  tokenizer snag for from-scratch models).
* **OpenAI-compatible server:** `python -m zivawhite.app serve`, then point any OpenAI client at `http://localhost:8000/v1`.
* **transformers:** `AutoModelForCausalLM.from_pretrained("exports/my-model")`.